### Yellow Taxi Trip Summaries 
This notebook calculates the daily and monthly summaries for the yellow taxi trips data.

In [0]:
# Import statements
from pyspark.sql.functions import count, max, min, avg, sum, round, date_format

In [0]:
# Read the data
df = spark.read.table("nyctaxi.02_silver.yellow_taxi_trips_enriched_silver")

In [0]:
# Daily summaries
df_daily_summary = df.\
    groupBy(df.tpep_pickup_datetime.cast("date").alias("pickup_date"),
            date_format("tpep_pickup_datetime", "yyyy-MM").alias("month_year")).\
    agg(
        count("*").alias("total_trips"),
        round(avg("passenger_count"), 1).alias("average_passengers"),
        round(avg("trip_distance"), 1).alias("average_distance"),
        round(avg("fare_amount"), 1).alias("average_fare_per_trip"),
        max("fare_amount").alias("maximun_fare"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )

In [0]:
# Monthly summary
df_monthly_summary = df.\
    groupBy(date_format("tpep_pickup_datetime", "yyyy-MM").alias("month_year")).\
    agg(
        count("*").alias("total_monthly_trips"),
        round(sum("total_amount"), 2).alias("total_monthly_revenue"),
        sum("passenger_count").alias("total_monthly_passengers"),
        round(sum("trip_distance"), 2).alias("total_monthly_distance")
                                            
    )

In [0]:
# Save daily summary table
df_daily_summary.write.mode("overwrite").saveAsTable("nyctaxi.03_gold.yellow_taxi_daily_summary_gold")

In [0]:
# Save monthly summary table
df_monthly_summary.write.mode("overwrite").saveAsTable("nyctaxi.03_gold.yellow_taxi_monthly_summary_gold")